In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv(r"../data/raw/orders.csv")
df.info()
df['order_date']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2030 entries, 0 to 2029
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   order_id       1954 non-null   object
 1   product_id     2030 non-null   object
 2   quantity       1980 non-null   object
 3   order_date     2030 non-null   object
 4   customer_city  2030 non-null   object
 5   payment_mode   2030 non-null   object
 6   order_status   2030 non-null   object
dtypes: object(7)
memory usage: 111.1+ KB


0        10-02-2025
1       Dec 25 2024
2        02/05/2025
3        2024-12-13
4       Dec 07 2024
           ...     
2025     01/12/2024
2026    Jan 06 2025
2027      15-Dec-24
2028     2025-02-13
2029     13-12-2024
Name: order_date, Length: 2030, dtype: object

In [35]:
df['customer_city'].unique()

array(['delhi', 'PUNE', 'Pune', 'Ahmedabad', 'mumbai', 'Delhi ', 'Delhi',
       'BENGALURU', 'chennai', 'Hyderabad', 'Mumbai', 'AHMEDABAD',
       'Bengaluru', 'hyderabad', 'bengaluru', 'Kolkata', 'ahmedabad',
       'pune', ' Delhi', 'Chennai ', 'Chennai', 'MUMBAI', ' Chennai',
       'Bombay', 'kolkata', ' Kolkata', 'HYDERABAD', 'Pune ',
       ' Hyderabad', 'CHENNAI', ' Bengaluru', ' Ahmedabad', ' Mumbai',
       'Madras', 'Hyderabad ', 'KOLKATA', 'DELHI', 'Mumbai ', 'Kolkata ',
       'Bangalore', 'Ahmedabad ', ' Pune', 'Calcutta', 'Bengaluru '],
      dtype=object)

In [36]:
# Standardize order_date and keep as string format
df['order_date'].astype("string").str.strip()
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce", dayfirst=False,format='mixed' )
print(df['order_date'].dtype)
df["order_date"] = df["order_date"].dt.strftime("%Y-%m-%d")

df.head(10)

datetime64[ns]


,order_id,product_id,quantity,order_date,customer_city,payment_mode,order_status
0,ORD2263,P009,1,2025-10-02,delhi,cash on delivery,Delivered
1,ORD1298,P003,10,2024-12-25,PUNE,CashonDelivery,Delivered
2,ORD1060,P002,8,2025-02-05,Pune,net banking,Cancelled
3,ORD2697,P002,9,2024-12-13,Ahmedabad,Cash on Delivery,Delivered
4,ORD1744,P008,2,2024-12-07,mumbai,Net Banking,Delivered
5,ORD2544,P033,3,2025-05-01,Ahmedabad,Cash on Delivery,Delivered
6,ORD1977,P002,1,2024-12-13,Delhi,upi,Delivered
7,ORD1353,p034,7,2025-01-29,mumbai,DebitCard,Delivered
8,ORD2432,P033,2,2025-01-22,Delhi,net banking,Delivered
9,ORD2402,P007,1,2024-12-18,BENGALURU,NET BANKING,Cancelled


In [37]:
df["order_id"] = df["order_id"].astype("string").str.strip()

# Remove duplicate rows based on order_id
df = df.drop_duplicates(subset=["order_id"], keep="first").reset_index(drop=True).copy()

# Standardize text fields
for col in ["customer_city", "payment_mode","order_status",'product_id']:
    df[col] = df[col].astype("string").str.replace("_", " ", regex=False).str.replace("-", "", regex=False).str.strip().str.title()

# Fill null values with Unknown in required columns
for col in ["order_date", "customer_city", "payment_mode"]:
    df[col] = df[col].fillna("Unknown").replace({"": "Unknown"})
    
df['quantity'] = (
    pd.to_numeric(df['quantity'], errors='coerce')  # convert safely
    .fillna(0)
    .astype(int)
    .abs()
)

df.head(10)

,order_id,product_id,quantity,order_date,customer_city,payment_mode,order_status
0,ORD2263,P009,1,2025-10-02,Delhi,Cash On Delivery,Delivered
1,ORD1298,P003,10,2024-12-25,Pune,Cashondelivery,Delivered
2,ORD1060,P002,8,2025-02-05,Pune,Net Banking,Cancelled
3,ORD2697,P002,9,2024-12-13,Ahmedabad,Cash On Delivery,Delivered
4,ORD1744,P008,2,2024-12-07,Mumbai,Net Banking,Delivered
5,ORD2544,P033,3,2025-05-01,Ahmedabad,Cash On Delivery,Delivered
6,ORD1977,P002,1,2024-12-13,Delhi,Upi,Delivered
7,ORD1353,P034,7,2025-01-29,Mumbai,Debitcard,Delivered
8,ORD2432,P033,2,2025-01-22,Delhi,Net Banking,Delivered
9,ORD2402,P007,1,2024-12-18,Bengaluru,Net Banking,Cancelled


In [38]:
print("Remaining duplicate order_id rows:", df["order_id"].duplicated().sum())
print("\nNull counts after cleaning:")
print(df[["order_date", "customer_city", "payment_mode"]].isna().sum())

Remaining duplicate order_id rows: 0

Null counts after cleaning:
order_date       0
customer_city    0
payment_mode     0
dtype: int64


In [39]:
print(df['customer_city'].unique())

<StringArray>
[    'Delhi',      'Pune', 'Ahmedabad',    'Mumbai', 'Bengaluru',   'Chennai',
 'Hyderabad',   'Kolkata',    'Bombay',    'Madras', 'Bangalore',  'Calcutta']
Length: 12, dtype: string


In [40]:
print(df['payment_mode'].unique())

<StringArray>
['Cash On Delivery',   'Cashondelivery',      'Net Banking',
              'Upi',        'Debitcard',       'Debit Card',
      'Credit Card',       'Netbanking',       'Creditcard',
               'Cc',               'Dc',            'U.P.I',
              'Cod']
Length: 13, dtype: string


In [41]:
print(df['order_status'].unique())

<StringArray>
['Delivered', 'Cancelled', 'Returned']
Length: 3, dtype: string


In [42]:
df[df['order_id'].isnull()]

,order_id,product_id,quantity,order_date,customer_city,payment_mode,order_status
26,<NA>,P023,2,2024-12-18,Mumbai,Upi,Cancelled


In [43]:
df = df.dropna(subset=['order_id'])

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1925 entries, 0 to 1925
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   order_id       1925 non-null   string
 1   product_id     1925 non-null   string
 2   quantity       1925 non-null   int64 
 3   order_date     1925 non-null   object
 4   customer_city  1925 non-null   string
 5   payment_mode   1925 non-null   string
 6   order_status   1925 non-null   string
dtypes: int64(1), object(1), string(5)
memory usage: 120.3+ KB


In [45]:
mapping = {
    'cashondelivery': 'Cash On Delivery',
    'cod': 'Cash On Delivery',

    'netbanking': 'Net Banking',

    'upi': 'UPI',

    'creditcard': 'Credit Card',
    'cc': 'Credit Card',

    'debitcard': 'Debit Card',
    'dc': 'Debit Card'
}

df['payment_mode_clean'] = (
    df['payment_mode']
    .str.lower()
    .str.replace(r'[\s\.]', '', regex=True)
)
df['payment_mode'] = df['payment_mode_clean'].map(mapping)
df.drop(columns=['payment_mode_clean'], inplace=True)
df['payment_mode'].unique()

array(['Cash On Delivery', 'Net Banking', 'UPI', 'Debit Card',
       'Credit Card'], dtype=object)

In [46]:
# mapping = {
#     'cashondelivery': 'Cash On Delivery',
#     'cod': 'Cash On Delivery',
#     'Cash On Delivery' : 'Cash On Delivery',

#     'Net Banking':'Net Banking',
#     'netbanking': 'Net Banking',

#     'Upi': 'UPI',
#     'U P I':'UPI',

#     'Credit Card':'Credit Card',
#     'creditcard': 'Credit Card',
#     'cc': 'Credit Card',

#     'Debit Card':'Debit Card',
#     'debitcard': 'Debit Card',
#     'dc': 'Debit Card'
# }


# df['payment_mode'] = df['payment_mode'].map(mapping)
# df['payment_mode'].unique()

In [47]:
df['customer_city'] = df['customer_city'].replace({'Madras': 'Chennai','Bengaluru':'Bangalore','Kolkata':'Calcutta'})
df['customer_city'].unique()

<StringArray>
[    'Delhi',      'Pune', 'Ahmedabad',    'Mumbai', 'Bangalore',   'Chennai',
 'Hyderabad',  'Calcutta',    'Bombay']
Length: 9, dtype: string

In [48]:
df['product_id'].unique()

<StringArray>
['P009', 'P003', 'P002', 'P008', 'P033', 'P034', 'P007', 'P035', 'P013',
 'P011', 'P022', 'P037', 'P006', 'P018', 'P021', 'P012', 'P015', 'P023',
 'P029', 'P036', 'P001', 'P040', 'P101', 'P016', 'P026', 'P032', 'P039',
 'P017', 'P014', 'P004', 'P028', 'P019', 'P005', 'P030', 'P027', 'P031',
 'P024', 'P025', 'P020', 'P010', 'P099', 'P038', 'P100']
Length: 43, dtype: string

In [ ]:
output_path = r"../data/processed/processed_orders_data.csv"
df.to_csv(output_path, index=False)

print(f"\nProcessed file saved to: {output_path}")
print("Final shape:", df.shape)


Processed file saved to: D:\Projects\Pandas - ZipMart Sales Intelligence Pipeline\processed_orders_data.csv
Final shape: (1925, 7)
